In [1]:
from delta import *
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName("Mining")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "2g")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark ready:", spark.version)

Spark ready: 3.3.0


In [2]:
from pyspark.ml.fpm import FPGrowth
from pyspark.sql import functions as F

basket = spark.read.format("delta").load(
    "/home/jovyan/work/delta_lake/gold/gold_basket"
)

# ✅ FIX: dedup items trong mỗi basket thành set (phòng trường hợp collect_list cũ)
basket = basket.withColumn("items", F.array_distinct(F.col("items")))

fp = FPGrowth(itemsCol="items", minSupport=0.01, minConfidence=0.1)
model = fp.fit(basket)

# ✅ FIX: cast array → JSON string thay vì .cast("string")
# vì Spark cast array thành [garden_tools, garden_tools] không có quotes
# → ast.literal_eval sẽ fail ở notebook sau
rules = model.associationRules \
    .withColumn("antecedent", F.to_json(F.col("antecedent"))) \
    .withColumn("consequent", F.to_json(F.col("consequent")))

rules.toPandas().to_csv(
    "/home/jovyan/work/delta_lake/gold/association_rules.csv",
    index=False
)

print("Saved!", rules.count(), "rules")
rules.orderBy(F.desc("lift")).show(10, truncate=False)

Saved! 29 rules
+------------------+------------------+-------------------+------------------+--------------------+
|antecedent        |consequent        |confidence         |lift              |support             |
+------------------+------------------+-------------------+------------------+--------------------+
|["perfumery"]     |["health_beauty"] |0.44               |4.610434782608696 |0.015214384508990318|
|["health_beauty"] |["perfumery"]     |0.15942028985507245|4.610434782608695 |0.015214384508990318|
|["home_confort"]  |["bed_bath_table"]|0.86               |3.1562436548223354|0.059474412171507604|
|["bed_bath_table"]|["home_confort"]  |0.2182741116751269 |3.1562436548223345|0.059474412171507604|
|["toys"]          |["baby"]          |0.38               |2.954193548387097 |0.02627939142461964 |
|["baby"]          |["toys"]          |0.20430107526881722|2.9541935483870967|0.02627939142461964 |
|["cool_stuff"]    |["baby"]          |0.30303030303030304|2.355816226783969 |0.0276